# IHDP and synthetic DGP exploration

Exploratory only. All reported results come from `experiments/run_experiment.py`.

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from src.data.ihdp import load_ihdp

split = load_ihdp(0)
print(split.meta)
print("true ATE (train):", round(split.train.ate_true, 4))

## Treatment imbalance and overlap

IHDP is imbalanced by construction: Hill (2011) removed a non-random subset
of treated units, leaving roughly 18% treated.

In [ ]:
from src.evaluation.diagnostics import overlap_diagnostics

diag = overlap_diagnostics(split.train)
pd.Series(diag).round(4)

In [ ]:
from src.evaluation.diagnostics import estimate_propensity

ps = estimate_propensity(split.train.x, split.train.t)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(ps[split.train.t == 0], bins=30, alpha=0.6, label="control", density=True)
ax.hist(ps[split.train.t == 1], bins=30, alpha=0.6, label="treated", density=True)
ax.set_xlabel("estimated propensity"); ax.set_ylabel("density")
ax.legend(); ax.set_title("Propensity overlap, IHDP realization 0")
plt.show()

## True effect heterogeneity

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(split.train.tau_true, bins=40)
ax.set_xlabel("true individual treatment effect"); ax.set_ylabel("count")
ax.set_title("Effect heterogeneity, IHDP realization 0")
plt.show()

## Synthetic DGP: the confounding knob

Increasing gamma concentrates propensities near 0 and 1, weakening overlap,
while the true ATE stays fixed at 1.0.

In [ ]:
from src.data.synthetic import SyntheticConfig, make_synthetic

rows = []
for gamma in [0.0, 1.0, 2.0, 3.0]:
    s = make_synthetic(SyntheticConfig(n_train=4000, n_test=100,
                                       confounding_strength=gamma, seed=0))
    d = overlap_diagnostics(s.train)
    rows.append({"gamma": gamma, "true_ate": round(s.train.ate_true, 3),
                 "smd_max": round(d["smd_max"], 3),
                 "ps_extreme": round(d["ps_frac_below_0.1"] + d["ps_frac_above_0.9"], 3)})
pd.DataFrame(rows)